In [ ]:
import pandas as pd
import numpy as np
import psycopg2
import time
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from datetime import datetime, timedelta

# --- КОНФИГУРАЦИЯ ---
DB_URL = "dbname=user43 user=user43 password=m5q3x8tpc7vn host=2.nntc.nnov.ru port=5402"
CSV_PATH = 'Модуль В/traffic_full_30min.csv'
HORIZON = 60 # Фиксированный горизонт по КЗ (60 минут)

In [ ]:
try:
    df = pd.read_csv(CSV_PATH)
    # Приводим к минутной интенсивности
    df['moment_cars'] = df['intensity_30min'] / 30.0 
    # Сдвиг таргета на 60 минут (предполагаем, что шаг в CSV - 30 минут, значит сдвиг на 2 строки)
    df['target'] = df['moment_cars'].shift(-2) 
    
    valid_data = df.dropna()
    X = valid_data[['moment_cars']]
    y = valid_data['target']
    
    # Обучение модели
    model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)
    model.fit(X, y)
    
    # Оценка точности
    preds = model.predict(X)
    mae = mean_absolute_error(y, preds)
    rmse = mean_squared_error(y, preds) ** 0.5
    r2 = r2_score(y, preds)
    
    print("-" * 40)
    print(f"✅ Модель обучена. Горизонт: +{HORIZON} мин")
    print("МЕТРИКИ КАЧЕСТВА:")
    print("MAE:  {0:7.2f}".format(mae))
    print("RMSE: {0:7.2f}".format(rmse))
    print("R2:   {0:7.2f}".format(r2))
    print("-" * 40)
    
except Exception as e:
    print(f"❌ Ошибка при обучении: {e}")
    exit()

In [ ]:
# 2. Прогнозирование
def predict_and_store():
    try:
        with psycopg2.connect(DB_URL) as conn:
            with conn.cursor() as cur:
                # Берем реальный факт за последние 5 минут для сглаживания
                cur.execute("""
                    SELECT COUNT(DISTINCT track_id), COALESCE(AVG(speed_km_h), 0) 
                    FROM full_tracking_data 
                    WHERE detection_time >= NOW() - INTERVAL '5 minute'
                """)
                row = cur.fetchone()
                
                # Защита от пустых ответов
                curr_count = float(row[0]) if row[0] is not None else 0.0
                curr_speed = float(row[1]) if row[1] is not None else 0.0
                
                now = datetime.now()
                target_time = now + timedelta(minutes=HORIZON)
                
                # Создаем DataFrame для предикта, чтобы не было Warning'ов
                input_df = pd.DataFrame([[curr_count]], columns=['moment_cars'])
                pred_val = float(model.predict(input_df)[0])
                pred_val = max(0.0, pred_val) # Интенсивность не может быть отрицательной
                
                # Вставка в БД
                cur.execute("""
                    INSERT INTO traffic_predictions 
                    (target_time, horizon_minutes, predicted_intensity, predicted_avg_speed) 
                    VALUES (%s, %s, %s, %s)
                """, (target_time, HORIZON, pred_val, curr_speed))
                
                print(f"[{now.strftime('%H:%M:%S')}] 🔮 Прогноз записан | Факт сейчас: {int(curr_count)} ТС | Прогноз через {HORIZON}м: {pred_val:.1f} ТС")
                
    except Exception as e:
        print(f"❌ Ошибка БД в цикле прогноза: {e}")

# Бесконечный цикл с обработкой прерывания
try:
    while True:
        predict_and_store()
        time.sleep(30) # Запуск по расписанию (раз в 30 секунд для демо)
except KeyboardInterrupt:
    print("\n🛑 Цикл прогнозирования остановлен пользователем.")